In [26]:
import pandas as pd
path = "../data/raw/cmu_books/booksummaries.txt"

df = pd.read_csv(path, sep="\t", header=None)
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

      0        1                                          2                3  \
0   620  /m/0hhy                                Animal Farm    George Orwell   
1   843  /m/0k36                         A Clockwork Orange  Anthony Burgess   
2   986  /m/0ldx                                 The Plague     Albert Camus   
3  1756  /m/0sww  An Enquiry Concerning Human Understanding       David Hume   
4  2080  /m/0wkt                       A Fire Upon the Deep     Vernor Vinge   

            4                                                  5  \
0  1945-08-17  {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...   
1        1962  {"/m/06n90": "Science Fiction", "/m/0l67h": "N...   
2        1947  {"/m/02m4t": "Existentialism", "/m/02xlf": "Fi...   
3         NaN                                                NaN   
4         NaN  {"/m/03lrw": "Hard science fiction", "/m/06n90...   

                                                   6  
0   Old Major, the old boar on the Manor Farm, ca...  


In [27]:
# from pathlib import Path
# import pandas as pd

ROOT = Path.cwd()

# Works if the notebook is opened from the project root
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

books_path = ROOT / "data" / "processed" / "books.parquet"

books = pd.read_parquet(books_path)

print("Dataset shape:", books.shape)
print("Columns:", books.columns.tolist())
print("Empty Genres: ", books["genres"].map(len).eq(0).sum())
print("Empty Authors: ", (books["author"] == "").sum())
print("Missing Dates: ", (books["publication_date"] == "").sum())


Dataset shape: (16551, 8)
Columns: ['wiki_id', 'freebase_id', 'title', 'author', 'publication_date', 'genres', 'summary', 'embedding_text']
Empty Genres:  3717
Empty Authors:  2379
Missing Dates:  5606


In [28]:
# reading 20 rows by hand
print(books.sample(20))


        wiki_id freebase_id                                  title  \
8104    7694517  /m/02693ng                       I Am a Barbarian   
441      156213   /m/014hj3                       The Nine Tailors   
3808    2595065   /m/07qhg_                                 Colony   
1761     872379   /m/03kfq5                               Papillon   
16008  31462253  /m/0gkxx6c                               The Drop   
3555    2354886   /m/0762tm                   Mendoza in Hollywood   
11766  16186787  /m/03wc9fd                        Into the Out Of   
3374    2192851   /m/06tw6m                 The Thief and the Dogs   
4711    3506780   /m/09h7dd                              The Lorax   
5086    3935667   /m/0b7kcg                        The Secret Room   
14501  24461375  /m/080q469                      The Black Bouquet   
15634  29660069  /m/0fpk6fw                               The Dead   
7242    6272268   /m/0fzz2c                         Naked in Death   
8780    8894751  /m/

In [29]:
print("Embedding text: ", books["embedding_text"].iloc[0])

Embedding text:  Title: Animal Farm
Author: George Orwell
Publication date: 1945-08-17
Genres: Children's literature, Fiction, Roman à clef, Satire, Speculative fiction
Summary: Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philosophy. The animals revolt and drive the drunken and irresponsible Mr Jones from the farm, renaming it "Animal Farm". They adopt Seven Commandments of Animal-ism, the most important of which is, "All animals are equal". Snowball attempts to teach the animals reading and writing; food is plentiful, and the farm runs smoothly. The pigs elevate themselves to positions of leadership and set aside special food items, ostensibly for their personal health. Napoleon takes the pups from the farm dogs and trains them privately. 

In [30]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
max_tokens = model.max_seq_length

tokenizer = model.tokenizer
texts = [
    tokenizer.decode(
        tokenizer(t, truncation=True, max_length=max_tokens)["input_ids"],
        skip_special_tokens=True,
    )
    for t in books["embedding_text"].tolist()
]

books_vectors = model.encode(
  texts,
  convert_to_numpy=True,
  normalize_embeddings=True,
  show_progress_bar=True
)

print(books_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/518 [00:00<?, ?it/s]

(16551, 384)
